# Farag (2023) — Tiny Matched-Filter 1D-CNN
**Paper:** M. M. Farag, "A Tiny Matched Filter-Based CNN for Inter-Patient ECG
Classification and Arrhythmia Detection at the Edge," *Sensors* 23(3):1365, 2023.

This notebook runs the full replication pipeline on **Colab (T4/A100)**, a **local GPU (RTX 4000 Ada / any CUDA)**, or **CPU**.

**Runtime → Change runtime type → T4 GPU** before running for best speed.

## 1 — Setup

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
print('Running on Colab:', IN_COLAB)

In [ ]:
# ── Option A: Google Drive (repo already on Drive) ────────────────────────────
# Mount Drive and point REPO_ROOT at your FYP folder.
# Change DRIVE_FYP_PATH to match your actual Drive layout.

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_FYP_PATH = '/content/drive/MyDrive/fyp'   # ← CHANGE THIS if needed
    if os.path.isdir(DRIVE_FYP_PATH):
        REPO_ROOT = DRIVE_FYP_PATH
        print('Using Drive repo at', REPO_ROOT)
    else:
        print('Drive path not found — will clone below')
        REPO_ROOT = None
else:
    # Local execution: repo is the parent of HasanML/paper3
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(''), '../..'))
    print('Local repo at', REPO_ROOT)

In [ ]:
# ── Option B: Clone from GitHub (if Drive mount failed or not applicable) ─────
# Uncomment and set your repo URL if cloning is preferred.

# GITHUB_URL = 'https://github.com/YOUR_USERNAME/fyp.git'
# if IN_COLAB and (REPO_ROOT is None or not os.path.isdir(REPO_ROOT)):
#     !git clone {GITHUB_URL} /content/fyp
#     REPO_ROOT = '/content/fyp'

PAPER3 = os.path.join(REPO_ROOT, 'HasanML', 'paper3')
print('paper3 dir:', PAPER3)
assert os.path.isdir(PAPER3), f'paper3 not found at {PAPER3}'

In [ ]:
# Install dependencies
# On Colab, TF is already installed — only the extras are needed.
import subprocess
req = os.path.join(PAPER3, 'requirements.txt')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--upgrade',
    'wfdb', 'tensorflow-model-optimization', 'xgboost', 'tqdm', 'threadpoolctl'
])
print('Dependencies installed.')

In [ ]:
# Set working directory to HasanML/paper3 so relative paths in configs work
os.chdir(PAPER3)
if PAPER3 not in sys.path:
    sys.path.insert(0, PAPER3)
print('Working directory:', os.getcwd())

## 2 — GPU / Device Check

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TF version :', tf.__version__)
print('GPU devices:', gpus)

if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    print('Memory growth enabled on', len(gpus), 'GPU(s)')
    # Optional: mixed precision for RTX / Ampere / Ada (~1.5-2x faster)
    # Uncomment the next two lines if you want float16 Tensor Core acceleration:
    # tf.keras.mixed_precision.set_global_policy('mixed_float16')
    # print('Mixed precision: mixed_float16 active')
else:
    print('No GPU found — training on CPU (slower)')

## 3 — Data

MIT-BIH is downloaded automatically from PhysioNet the first time it is needed  
(via `wfdb.dl_database`). The download is ~100 MB and takes ~1 minute on Colab.

To persist across Colab sessions, the data is written to your Drive:
`<DRIVE_FYP_PATH>/datasets/mitbih/mitdb/`

If the data is already there, nothing is downloaded.

In [ ]:
import wfdb
from pathlib import Path

# Local mirror: either on Drive (persistent) or in /content (lost on reset)
MITDB_DRIVE = Path(REPO_ROOT) / 'datasets' / 'mitbih' / 'mitdb'
MITDB_LOCAL = Path(PAPER3)   / 'data' / 'mitdb'   # paper3 data_dir

MITDB_DRIVE.mkdir(parents=True, exist_ok=True)
MITDB_LOCAL.mkdir(parents=True, exist_ok=True)

# Records used by the replication (44 of 48; paced records excluded)
ALL_RECORDS = [
    '100','101','103','105','106','108','109','111','112','113','114','115',
    '116','117','118','119','121','122','123','124','200','201','202','203',
    '205','207','208','209','210','212','213','214','215','217','219','220',
    '221','222','223','228','230','231','232','233','234',
]

missing = [r for r in ALL_RECORDS
           if not (MITDB_DRIVE / f'{r}.hea').exists()]

if missing:
    print(f'Downloading {len(missing)} records from PhysioNet...')
    wfdb.dl_database('mitdb', dl_dir=str(MITDB_DRIVE),
                     records=missing, annotators=['atr'])
    print('Download complete.')
else:
    print(f'All {len(ALL_RECORDS)} records already cached at', MITDB_DRIVE)

## 4 — Tests (34/34 expected)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short', '-q'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## 5 — Sanity Check (~2 min)

RR-only XGBoost: expected acc ~93 %, N ~97 %, S ~60 %, V ~58 %

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'src.run_experiment', '--rr-baseline'],
    capture_output=False, text=True
)
print('Exit code:', result.returncode)

## 6 — Train P1 (Model 6, 10 seeds)

Expected targets on DS2:
- OvR accuracy: **98.18 %**
- Avg F1: **92.17 %**  
- SVEB sensitivity (Se_S): **85.3 %**  
- VEB sensitivity  (Se_V): **96.34 %**

GPU: ~5 min &nbsp;|&nbsp; CPU: ~20–60 min

Interrupted runs resume automatically (each seed is cached).

In [ ]:
# Select device: 'auto' picks GPU if available, falls back to CPU
DEVICE = 'auto'   # or 'cpu' | 'cuda'

# Optional: mixed_float16 for RTX / Ampere / Ada (~1.5-2x faster on GPU)
MIXED = '--mixed-precision' if gpus else ''

cmd = [sys.executable, '-m', 'src.run_experiment',
       '--p1', '--device', DEVICE]
if MIXED:
    cmd.append(MIXED)

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=False, text=True)
print('Exit code:', result.returncode)

## 7 — Evaluate Results

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'src.evaluate', '--run', 'P1', '--per-seed'],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'src.evaluate', '--run', 'P1', '--confusion'],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
# Show the 13 matched-filter template plots
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

template_path = 'results/figures/templates.png'
if os.path.exists(template_path):
    img = mpimg.imread(template_path)
    plt.figure(figsize=(14, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('13 Matched-Filter Templates (paper Figure 4 equivalent)')
    plt.tight_layout()
    plt.show()
else:
    print('Templates not found. Run: python -m src.run_experiment --templates-only')

In [ ]:
# Show confusion matrix
cm_path = 'results/figures/confusion_P1.png'
if os.path.exists(cm_path):
    img = mpimg.imread(cm_path)
    plt.figure(figsize=(6, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('P1 Confusion Matrix (DS2)')
    plt.tight_layout()
    plt.show()

## 8 — Binary Extension (optional)

P2 = binary AAMI (N vs S,V,F,Q)  
P3 = binary raw  (symbol N only vs everything else)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'src.run_experiment',
     '--p1', '--p2', '--p3', '--device', DEVICE],
    capture_output=False, text=True
)
print('Exit code:', result.returncode)

## 9 — TFLite Export for ESP32 (optional)

Requires P1 to be trained first.

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'src.run_experiment', '--p1', '--export'],
    capture_output=False, text=True
)
print('Exit code:', result.returncode)

# List exported files
export_dir = Path('results/export')
if export_dir.exists():
    for f in sorted(export_dir.iterdir()):
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name:<50} {size_kb:6.1f} KB')

## 10 — Save All Metrics to CSV

In [ ]:
import pandas as pd

result = subprocess.run(
    [sys.executable, '-m', 'src.evaluate', '--csv', 'results/all_metrics.csv'],
    capture_output=True, text=True
)
csv_path = 'results/all_metrics.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(df.to_string())